# 負面新聞多標籤分類模型

此 notebook 使用 TF-IDF 特徵提取和 PyTorch 多層感知機進行多標籤文本分類。

## 安裝所需套件

In [ ]:
import sys
import subprocess

# 安裝所需的套件
packages = ['pandas', 'openpyxl', 'torch', 'scikit-learn', 'joblib']
for package in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', package])

## 載入與查看資料

In [ ]:
import google.colab
from google.colab import drive
import pandas as pd

# 輸入與查看資料
drive.mount("/content/drive")
negative_news_path = "/content/drive/MyDrive/2025-07-09-negative_news.xlsx"
news = pd.read_excel(negative_news_path)
print("Data shape:", news.shape)
print(news.head())

# 定義關鍵詞
key_word = [
    "洗黑錢", "貪污", "賄賂", "內幕交易", "毒品", "詐欺", 
    "走私", "逃稅", "操控市場", "有組織罪行", "舞弊", "電騙", "金融犯罪"
]
print(f"\n共有 {len(key_word)} 個關鍵詞")

## 載入標註資料

In [ ]:
import pandas as pd

file_path = "/content/多標籤標註_補齊語意相近.xlsx"
df = pd.read_excel(file_path)

print(f"資料集大小: {df.shape}")
print(f"\n前幾筆資料:")
print(df.head())

# 檢查標籤欄位
label_cols = df.select_dtypes(include='number').columns
print(f"\n標籤欄位: {list(label_cols)}")

## 特徵提取 - TF-IDF 向量化

In [ ]:
import torch
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.utils.validation import check_is_fitted
import joblib

# 提取文本特徵
corpus = df['新聞內容'].astype(str).apply(lambda x: x.strip())
print(f"語料庫大小: {len(corpus)} 篇文章")

# 建立 TF-IDF 向量化器
vectorizer = TfidfVectorizer(max_features=1000, ngram_range=(1, 2))
vectorizer.fit(corpus)

# 轉換文本為特徵矩陣
X = vectorizer.transform(corpus)
Y = df.select_dtypes(include="number")

print(f"特徵矩陣形狀: {X.shape}")
print(f"標籤矩陣形狀: {Y.shape}")

# 轉換為 PyTorch 張量
X_tensor = torch.from_numpy(X.toarray()).float()
Y_tensor = torch.from_numpy(Y.to_numpy()).float()

print(f"X_tensor 形狀: {X_tensor.shape}")
print(f"Y_tensor 形狀: {Y_tensor.shape}")

# 保存 vectorizer
joblib.dump(vectorizer, 'tfidf_vectorizer.pkl')
check_is_fitted(vectorizer)
print("\nVectorizer 已保存")

## 分割訓練集與測試集

In [ ]:
from sklearn.model_selection import train_test_split

# 分割訓練集與測試集
X_train, X_test, Y_train, Y_test = train_test_split(
    X_tensor, Y_tensor, 
    test_size=0.1, 
    random_state=42
)

print(f"訓練集大小: {X_train.shape[0]} 筆")
print(f"測試集大小: {X_test.shape[0]} 筆")
print(f"特徵維度: {X_train.shape[1]}")
print(f"標籤數量: {Y_train.shape[1]}")

## 建立多層感知機模型

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

# 定義多層感知機模型
class MLP_MultiLabel(nn.Module):
    """多標籤分類用的多層感知機"""
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 32)
        self.fc4 = nn.Linear(32, 16)
        self.out = nn.Linear(16, output_dim)
        
        # Dropout 防止過擬合
        self.dropout = nn.Dropout(p=0.2)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = self.dropout(x)
        x = F.relu(self.fc3(x))
        x = self.dropout(x)
        x = F.relu(self.fc4(x))
        x = self.dropout(x)
        return self.out(x)

# 初始化模型
input_dim = X_tensor.shape[1]
output_dim = Y_tensor.shape[1]
model = MLP_MultiLabel(input_dim=input_dim, output_dim=output_dim)

print(f"模型已建立")
print(f"輸入維度: {input_dim}")
print(f"輸出維度: {output_dim}")
print(f"\n模型架構:")
print(model)

## 訓練模型

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

# 設置訓練參數
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.BCEWithLogitsLoss()  # 適用於多標籤分類
train_loader = DataLoader(
    TensorDataset(X_train, Y_train), 
    batch_size=10, 
    shuffle=True
)

print(f"使用設備: {device}")
print(f"批次大小: 10")
print(f"訓練回合: 50\n")

# 訓練模型
num_epochs = 50
losses = []

for epoch in range(num_epochs):
    epoch_loss = 0
    for batch_idx, (x, y) in enumerate(train_loader):
        x, y = x.to(device), y.to(device)
        
        model.train()
        optimizer.zero_grad()
        
        # 前向傳播
        logits = model(x)
        loss = criterion(logits, y)
        
        # 反向傳播
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(train_loader)
    losses.append(avg_loss)
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}")

print(f"\n訓練完成!")

## 模型評估

In [ ]:
from sklearn.metrics import classification_report

# 評估模型
with torch.no_grad():
    model.eval()
    X_test_device = X_test.to(device)
    logits = model(X_test_device)
    
    # 將邏輯回歸轉換為二元預測
    y_pred = (torch.sigmoid(logits) > 0.5).int().cpu().numpy()
    y_true = Y_test.numpy()

# 生成分類報告
report = classification_report(
    y_true, y_pred,
    zero_division=0,
    output_dict=True
)

# 轉換為 DataFrame 並保存
report_df = pd.DataFrame(report).transpose()
report_df.to_excel("多標籤分類_F1評估報告.xlsx")

print("\n分類報告已保存至 '多標籤分類_F1評估報告.xlsx'")
print("\n核心指標:")
print(report_df[['precision', 'recall', 'f1-score']].round(4))

## 檢視評估報告

In [ ]:
file_path1 = "多標籤分類_F1評估報告.xlsx"
report = pd.read_excel(file_path1)
print("評估報告:")
print(report)

## 保存模型

In [ ]:
# 保存完整模型
torch.save(model, 'truemodel.pth')
print("模型已保存至 'truemodel.pth'")

# 也保存模型的狀態字典
torch.save(model.state_dict(), 'model_state_dict.pth')
print("模型狀態已保存至 'model_state_dict.pth'")